# First exercise submission from real CS1 IDE activity

The target is whether an exercise receives its first recorded submission in the next ten minutes. The inputs are counts from the preceding two minutes of editor activity for the same learner and exercise. A submission here means the student sent code to assessment. Its grading result is outside the dataset's event labels.

Source: Juho Leinonen and Arto Hellas, *IDE Action Log Dataset from a CS1 MOOC*, version 1, 2022, CC BY 4.0. This run uses fixed byte ranges spread across the release CSV and retains interior learner blocks. The sample can favor learners whose event blocks fit inside a range. The notebook aggregates events and displays counts only. [Zenodo release](https://doi.org/10.5281/zenodo.6903968) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    PrecisionRecallDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_predict

from cs1_submission_data import (
    HORIZON_SECONDS, WINDOW_SECONDS, fetch_source, load_windows, sample_checksum
)
from keystroke_data import FEATURE_NAMES, SEED
from keystroke_model import build_model, export_model, score_windows

source = fetch_source()
windows, counts = load_windows(source)
x = windows.select(*FEATURE_NAMES).to_numpy()
y = windows["submit_within_horizon"].to_numpy()
groups = windows["student"].to_numpy()
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
development, holdout = next(split.split(x, y, groups))
assert set(groups[development]).isdisjoint(groups[holdout])
assert np.isfinite(x).all()

print(json.dumps({
    **counts,
    "sample_sha256": sample_checksum(source),
    "feature_window_seconds": WINDOW_SECONDS,
    "prediction_horizon_seconds": HORIZON_SECONDS,
    "development_learners": len(set(groups[development])),
    "holdout_learners": len(set(groups[holdout])),
    "holdout_windows": len(holdout),
    "holdout_positive_windows": int(y[holdout].sum()),
}, indent=2))

## Look back two minutes and check the next ten

The feature history is `[T − 120 seconds, T)`. It covers inserts and removals, bins with edits, the longest edit pause, the fraction of edits that removed text, and pastes. Candidate windows have at least one insert, removal, or paste. Windows cover time before the exercise's first recorded submission. A prediction is evaluated only when the part deadline and the last observed IDE activity for that learner and part are at least ten minutes beyond the anchor.

The target checks the first submission for the same exercise in `[T, T + 600 seconds)`. Since logging was optional and could be interrupted, an observed ten-minute horizon can still have missing events. We keep entire learners out of the holdout and select the operating threshold on grouped out-of-fold development scores.

In [ ]:
folds = GroupKFold(n_splits=4)
development_scores = cross_val_predict(
    build_model(), x[development], y[development], groups=groups[development],
    cv=folds, method="predict_proba",
)[:, 1]
precision, recall, thresholds = precision_recall_curve(y[development], development_scores)
denominator = precision[:-1] + recall[:-1]
f1 = np.divide(
    2 * precision[:-1] * recall[:-1],
    denominator,
    out=np.zeros_like(thresholds),
    where=denominator > 0,
)
threshold = float(thresholds[np.argmax(f1)])
model = build_model().fit(x[development], y[development])
print(f"Development-selected F1 threshold: {threshold:.6f}")

In [ ]:
scores = model.predict_proba(x[holdout])[:, 1]
flagged = scores >= threshold
truth = y[holdout]
active_bins = x[holdout, FEATURE_NAMES.index("active_edit_bins")]
tn, fp, fn, tp = confusion_matrix(truth, flagged, labels=[0, 1]).ravel()
print(json.dumps({
    "average_precision": round(average_precision_score(truth, scores), 4),
    "active_edit_bins_average_precision": round(average_precision_score(truth, active_bins), 4),
    "constant_score_average_precision": round(float(truth.mean()), 4),
    "precision_at_threshold": round(precision_score(truth, flagged, zero_division=0), 4),
    "recall_at_threshold": round(recall_score(truth, flagged, zero_division=0), 4),
    "f1_at_threshold": round(f1_score(truth, flagged, zero_division=0), 4),
    "true_positive": int(tp), "false_positive": int(fp),
    "true_negative": int(tn), "false_negative": int(fn),
}, indent=2))

fig, ax = plt.subplots(figsize=(6.2, 4.0))
PrecisionRecallDisplay.from_predictions(truth, scores, name="Logistic regression", ax=ax)
PrecisionRecallDisplay.from_predictions(truth, active_bins, name="Active edit bins", ax=ax)
ax.axhline(truth.mean(), color="gray", linestyle="--", label="Positive prevalence")
ax.set_title("First submission on held-out learners from the CS1 sample")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## What the score measures

The score estimates a short-term event for this historical course and the selected population of observed editor windows. Independent evidence is needed for learning progress, authorship, and course outcomes. A learner who edits slowly, plans away from the IDE, or disables logging can leave a different trace from someone who works entirely in the editor. Treat this as an offline baseline.

In [ ]:
path = Path("cs1-submission-model.npz")
export_model(model, path, threshold=threshold)
reloaded, reloaded_threshold = score_windows(x[holdout], path)
np.testing.assert_allclose(reloaded, scores, atol=1e-12, rtol=0)
assert reloaded_threshold == threshold
print(f"Exported {path.name} ({path.stat().st_size:,} bytes).")
print(f"Reloaded artifact reproduced all {len(holdout)} held-out scores.")